# Processing Metabolomic NMR Data -- A Practical Guide
*Marcel Utz, KIT, 2026*

## Contents
1. Basics and Mathematical Foundation
2. Processing Spectra with `NMRlab`
3. Quantitative Information from Series of Spectra

## 1 Basics and Mathematical Foundation

The goal of metabolomic NMR data analysis is to determine the composition of the mixture under study.
That sounds simple enough, but requires us to contemplate what we do and typically do not know
about our sample first.

Metabolic mixtures are solutions, most often in an aqueous medium, of a large variety of
organic molecules which are either the products or the educts of physiological processes.
They typically contain
- a solvent (often water)
- inorganic ions
- small organic molecules
- macromolecules (proteins, polysaccharides)
- lipid aggregates (micelles and liposomes)
- cells (live and dead)

What we typically are after are the *concentrations* of a subset of the organic
molecules present. 

However, one may analyse metabolic mixtures with a different agenda. For example,
the aim may be to discover novel or unexpected molecules, which may be present at
low concentration, or substances that are somewhat intermediate between macromolecules
and small molecules, such as peptides, nucleic acids, or saccharides of various lengths
and topologies.

In an abstract mathematical sense, a complex mixture is characterised of
a *solvent* $s_0$ in combination with a list of dissolved species $s_k$, $1\le k \le n$.
A complete description of the mixture can be given by a vector of amounts $n_k>0$, $0\le k$.
The aim of metabolic analysis is to determine some or all of these amounts from an experimental
spectrum.

A *spectrum* is the result of an experiment that exposes the complex mixture to some
kind of radiation, and produces a function $S(\omega)$, where $\omega$ is the spectral
variable. In the case of NMR, $\omega$ is usually a frequency or a chemical shift value. 
For other techniques, $\omega$ may represent a different physical quantity, such as the $m/Z$ value
in the case of mass spectrometry, or a wavelength or wavenumber in the case of optical or
IR spectroscopy.

The basic idea is that the spectrum depends on the amounts in a known manner, such
that the amounts can be determined once the spectrum has been measured. Mathematically,
we can regard the measured spectrum as a function not only of $\omega$, but of the amounts
$n_k$, as well:
$$ S(\omega) = S(\{n_k\};\omega). $$
In reality, the spectrum we observe is *not* fully determined by the amounts. We need to
allow for experimental noise, which adds a stochastic component to the spectrum:
$$ S_\text{obs}(\omega) = S(\{n_k\};\omega)+ \tilde N(\omega),$$
where the tilde signals that $N(\omega)$ is a stochastic variable.

The problem of analysing a complex mixture can therefore be formulated as follows:

> Determine the amounts ${n_k}$ such that the observed spectrum is represented
> as closely as possible by the function $S({n_k};\omega)$. 

Of course, in practice, this presumes that we have a way of calculating the spectrum
for different combinations of $\{n_k\}$. Even if we do, finding the right combination
of $n_k$ is a typical inverse problem. Such problems are notoriously hard to solve in
general - and that is assuming that we know in
advance which compounds $s_k$ to expect in the mixture. In most practical cases, there
may be compounds present that we do not know. Solving the problem above is therefore
notoriously *difficult*.

### Partial Spectra and the Gibbs-Duhem Equation

The *partial spectrum* $S_k(\{n_k\},\omega)$ is defined as
$$
S_k(\{n_k\},\omega) = \left(\frac{\partial S({n_k};\omega) }{\partial n_k}\right)_{n_{j\ne k};\omega}. 
$$
In NMR, we can assume that the spectrum is *homogeneous* of degree 1 in the amounts $n_k$. This basically means
that the doubling any of the amounts will increase the contribution of the compound in question to the
spectrum proportionally. Formally, we have then
$$ 
S_k(\lambda \{n_k\},\omega) = \lambda S_k(\{n_k\};\omega).
$$ 
It can be shown by using Euler's homogeneous function theorem that in this case, the spectrum is an additive superposition of the 
partial spectra:
$$
S_k(\{n_k\},\omega) = \sum n_k\, S_k(\{n_k\};\omega).
$$
This is, by the way, analogous to the famous Gibbs-Duhem equation of Thermodynamics, which
relates the amounts and chemical potentials of compounds in mutual thermodynamic equilibrium.
If, on top of this, we can assume that the partial spectra for all compounds are indendent of
the amounts present, we arrive at the simple additive formula
$$
S(\{n_k\},\omega) = \sum n_k\, S_k(\omega).
$$
In this case, the spectrum is a very simple additive superposition of mutually independent 
contributions from each compound present. To a good approximation, this is the 
case for NMR spectra in the context of metabolomics.

> NOTE: 
> We have deliberately used the amounts and *not* the concentrations. The argument is based on 
> scaling of the *entire* sample, including the solvent. This means that the scaling
> conserves the concentrations of all species in the sample, and therefore does 
> not affect things such as pH etc.






## 2 Processing NMR Data

We use the `NMRlab.jl` package to process a sequence of NMR spectra. The package is under development by Manaz, but is very close
to its first public release. Documentation can be found at [https://abragam.imt.kit.edu/group/nmrlab-docs/](https://abragam.imt.kit.edu/group/nmrlab-docs/index.html).

> NOTE: "NMRlab" is a working name. For the public release, we plan to change the name, since the original one is already taken by an unrelated project.

In [ ]:
import Pkg
Pkg.activate(".")
Pkg.resolve()
# Pkg.add("Revise")
#Pkg.instantiate()
using Revise
#Pkg.rm("NMRlab")
#Pkg.develop(path="/Users/mu3q/Dropbox/Source/NMRlab.jl")
#Pkg.add(url="https://github.com/marcel-utz/NMRlab.jl")

using NMRlab
using Plots

default(size=(600,400),legendfontsize=8,guidefontsize=10,tickfontsize=8,fmt=:png,dpi=100)

As explained in the documentation, `NMRlab` comes with some example data sets. Here is a plot of a Bruker FID:

In [ ]:
using NMRlab
using NMRlab.Examples
using Plots: plot, savefig

data_bruker = NMRlab.Examples.Data["HCC cell culture media spectra"]

# High level vendor loader (recommended)
params_bruker, data_td = NMRlab.load(joinpath(data_bruker["path"], "10"), :Bruker)

t = data_td.coord[1]
y = real.(data_td.dat)

plot(t, y;
xlabel = "time / s",
ylabel = "signal (a.u.)",
title = "Bruker FID (real part)")

### Accessing Demonstration Data
Yaping has kindly provided a set of demonstration data files for us to use in this demo. They consist of three repeats of a 1D NMR experiment on  the supernatant of a cell culture
    taken at different time points. The data is stored in JEOL format.
Julia offers convenient tools to traverse the file system:

In [ ]:
for (path, dirs, files) in walkdir("./NMR data --monolayer")
    print("Directory: ", path, "\n")
    for file in files
        print("  File: ", joinpath(path,file), "\n")
    end
end 

We can use this to read all data and assemble it into a `SpectData` object.

In [ ]:
fids=[]
fnames=[]

for (path, dirs, files) in walkdir("./NMR data --monolayer")
    print("Directory: ", path, "\n")
    for file in files
        print("  File: ", joinpath(path,file), "\n")
        params,data=NMRlab.load(joinpath(path,file), :JEOL)
        push!(fids, data) 
        push!(fnames, basename(file))
    end
end 

fidData=SpectData(hcat(fids...),(NMRlab.coords(fids[1],1) ,fnames))  # combine all FIDs into a single data object

We now have a two-dimenisonal `SpectData` object with the time as the first axis and the file name as the second:

In [ ]:
NMRlab.coords(fidData,2)  # check that the file names are stored as 2nd dimension coordinates

We need to process this raw time-domain data to obtain spectra. For this, we define
a processing stack:

In [ ]:
process = Chain(
    ZeroFill([2^16,:]),
    Apodize([0.5pi,:]),
    FourierTransform([2^16,18],[1]),
    x-> x[10000:end-10000,:],  # crop the spectrum to the region of interest
    PhaseCorrect(0.4pi,2pi*0.00172,1),
    NMRlab.AutoPhaseCorrectChen(1,verbose=false,γ=0.0e-5),
    # MedianBaselineCorrect(1, wdw=1000)
)

processedData = process(fidData)  # apply the processing chain to the FIDs

n=1
plot(NMRlab.coords(processedData,1)./600 .+ 4.78, real.(processedData[:,n]);
xlabel = "chemical shift (ppm)",
ylabel = "signal (a.u.)",
xaxis = :flip,
title = NMRlab.coords(processedData,2)[n], xlims=(-0.1,9.0),ylims=1e4 .* (-0.1,1.05)) 

In [ ]:
# processedData[32500:33500,:] .= 0.0
a=plot()  # create an empty plot object to hold all spectra
for n in 1:size(processedData,2)    
    plot!(a,NMRlab.coords(processedData,1)./600 .+ 4.78, 200n .+ real.(processedData[:,n]);
        xlabel = "chemical shift (ppm)",
        ylabel = "signal (a.u.)",
        xaxis = :flip,
        ylims=5e3 .* (-0.1,1.05),
        legend=false)
end
savefig(a, "processed_spectra.svg")
a

In [ ]:
ELandScape=[ processedData[:,9] |> PhaseCorrect(x1,x2,1) |> Derivative(1) |> NMRlab.entropy
     for x1 in range(-pi/2,pi/2,length=40), 
         x2 in range(-0.005,0.005,length=40) ]
Plots.contourf(range(-pi/2,pi/2,length=40), range(-0.005,0.005,length=40), ELandScape';
     xlabel="zero order phase / rad", 
     ylabel="first order phase / rad s", 
     title="entropy landscape", 
     colorbar=false, 
     size=(600,600),)   

In [ ]:
processedData = processedData |> MedianBaselineCorrect(1, wdw=10000) # apply baseline correction to the processed data

In [ ]:
a=plot()  # create an empty plot object to hold all spectra
for n in 1:size(processedData,2)    
    plot!(a,NMRlab.coords(processedData,1)./600 .+ 4.78, 5000*(n-1) .+ processedData[:,n] ;
        xlabel = "chemical shift (ppm)",
        ylabel = "signal (a.u.)",
        size=(800,1200),
        xaxis = :flip,
        ylims=5e4 .* (-0.1,2.05),
        legend=false)
end
a

## Obtaining Partial Spectra

In order to determine amounts and concentrations, we need the partial spectra of the
compounts of interest. We make the assumption discussed above that the partial spectra
are independent of the amounts. Such partial spectra can be obtained either by 
measurement or by simulation. The former has the advantage that the spectral parameters
do not need to be known in advance. By contrast, experimental spectra are necessarily
measured under specific conditions (solvent, temperature, pH, magnetic field strength) 
that may not correspond to the ones in the measured spectrum. 

A useful compromise is the GISSMO database, which contains the spectral parameters
(chemical shifts and $J$-couplings) for thousands of small molecules. These data have
obviously been derived from experiments, but they allow spin dynamic simulation of the 
partial spectrum at any field strength.


The GISSMO database entries have a unique identifier. To find the identifier corresponding to a specific metabolite, we can use the `GISSMO.search` function:

In [ ]:
import NMRlab.GISSMO

metabolite_names = ["Glucose", "L-lactate", "L-Alanine", "L-Glutamine", "L-Valine","L-Phenylalanine"]
gissmo_hits = [GISSMO.search(metabolite) for metabolite in metabolite_names]

Some of these names produced multiple hits. In this case, it makes sense to choose the right one manually. For simplicity, we just use the first hit in every case for the present demo.

In [ ]:
metabolites=[first(k) for k in gissmo_hits]

The GISSMO database contains a spin matrix for each entry. This is a square matrix of dimension $n$, where $n$ is the number of protons
in the molecule that contribute to the NMR spectrum. The diagonal contains the chemical shifts, whereas $J$-couplings between protons
are listed in the off-diagonal elements. Here is glucose as an example:

In [ ]:
SpM=GISSMO.SpinMatrix(metabolites[1]["id"])

The Glucose entry contains the spin matrix for both $\alpha$ and $\beta$-Glucose. Since there are no
$J$-couplings between the two different molecules, it makes no sense to simulate the entire
14-spin system together. We therefore separate the two out, resulting in two $7\times 7$ spin matrices:

In [ ]:
n,H=GISSMO.Hamiltonian(SpM[8:14,8:14],freq=600.0, ctr=4.78)
Plots.spy(H)

In [ ]:
using LinearAlgebra
using SparseArrays
using ProgressMeter

# δt = π/2 / opnorm(H,Inf)
δt = step(NMRlab.coords(fidData,1))
l = length(NMRlab.coords(fidData,1))

# compute the progagator
P=SpinSim.expm(im*δt*(H))
droptol!(P,1.0e3*eps(Float64))
rho0=sum(SpinSim.SpinOp(n,SpinSim.Sx,k) for k=1:n)
Fp=sum(SpinSim.SpinOp(n,SpinSim.Sp,k) for k=1:n)
fid = ComplexF64[]
sizehint!(fid,l)
ρ = rho0

@showprogress for k=1:l  
    push!(fid, tr(ρ*Fp))
    ρ = P*ρ*P';
    droptol!(ρ,10*eps(Float64))
end

simfid = SpectData(fid, (NMRlab.coords(fidData,1),))

In [ ]:
procSim = Chain(
    ZeroFill([2^16]),
    Apodize([2.5pi]),
    x-> begin x[1]*=0.5; x end,
    FourierTransform([2^16],[1]),
    x->x[10000:end-10000],
)

simspect = procSim(simfid)
plot(NMRlab.coords(simspect,1)./600 .+ 4.78, real.(simspect[:,1]),xaxis=:flip, xlims=(3,5.5),)

## Converting a list of metabolites to a decomposition matrix

In practice, we have a list of metabolites that we would like to use to obtain quantitative information (amounts) from
our experimental NMR spectra. In the following, we will generate the partial spectra, assemble a decomposition matrix, and save it in a file
so we can re-use it without having to simulate the partial spectra again.

### List of Hamiltonians
As a first step, we generate a list of metabolites and their associated spin Hamiltonians. For some of them, we can
directly use the information from the GISSMO database. In some cases, however, we may need to either modify the GISSMO spin matrix, as in the case of Glucose. In other cases, GISSMO may not contain an entry for a compound that we need (e.g., DMSO).

To this effect, we create a dictionary that maps the metabolite name to the Hamiltonian matrix.

In [ ]:
n,H=GISSMO.Hamiltonian(SpM[8:14,8:14],freq=600.0, ctr=4.78)
Hamiltonians = Dict{String,Any}("α-Glucose"=>H) 

The other half of the GISSMO entry for Glucose is the spin matrix for $\beta$-Glucose:

In [ ]:
n,H=GISSMO.Hamiltonian(SpM[1:7,1:7],freq=600.0, ctr=4.78)
push!(Hamiltonians, "β-Glucose"=>H)

Another entry we will need is the one for TSP. We assume that it consists of a single spin at zero ppm. This gives a single peak; we will have to take into account that the real TSP molecule has 9 spins. The "amount" of TSP that we will determine with our approach will be a factor of 9 greater than the actual TSP content in the sample.

In [ ]:
n,H=GISSMO.Hamiltonian(zeros(1,1),freq=600., ctr=4.78)
push!(Hamiltonians, "TSP"=>H)

Now, we can add the Hamiltonians of the metabolites from the GISSMO database:

In [ ]:
for k=2:length(metabolites)
    n,H=GISSMO.Hamiltonian(metabolites[k]["id"],freq=600.0, ctr=4.78)
    push!(Hamiltonians, metabolites[k]["name"]=>H)
end
Hamiltonians

### Simulating the partial spectra

Now that we have a list of the relevant Hamiltonians, we need to use the above quantum dynamics algorithm to simulate the partial spectra.

In [ ]:
function SimSpect(H,δt,l)
    n=convert(Int, log2(size(H,1))) # size of the spin system is 2^n, so we can compute n as the log2 of the matrix size
    P=SpinSim.expm(im*δt*(H))
    droptol!(P,1.0e3*eps(Float64))
    rho0=2.0^(-n)*sum(SpinSim.SpinOp(n,SpinSim.Sx,k) for k=1:n)
    Fp=sum(SpinSim.SpinOp(n,SpinSim.Sp,k) for k=1:n)
    fid = ComplexF64[]
    sizehint!(fid,l)
    ρ = rho0

    @showprogress for k=1:l  
        push!(fid, tr(ρ*Fp))
        ρ = P*ρ*P';
        droptol!(ρ,10*eps(Float64))
    end
    SpectData(fid, (range(0.0,step=δt,length=l),))
end

In [ ]:
partialSpectra = Dict{String,Any}()

for (name, H) in Hamiltonians
    println("Simulating spectrum for ", name)
    partialSpectra[name] = SimSpect(H,δt, length(NMRlab.coords(fidData,1))) |> procSim
end



In [ ]:
a=plot()  # create an empty plot object to hold all spectra
for (k,(name, spect)) in enumerate(partialSpectra )
    plot!(a,NMRlab.coords(spect,1)./600 .+ 4.78, real.(spect[:,1]).+(k-1)*5e2,
    xaxis=:flip, ylims=6.0e3.*[-0.1,1.0], label=name,legend=:topleft,size=(800,600))
end
a

Now that we have laboriously computed these partial spectra, we may want to save them in a file so we can retrieve them later when needed. There are several methods in Julia to save binary and structured objects. We use the `JLD2` package here.

In [ ]:
using JLD2

In [ ]:
partialSpects= SpectData(hcat(values(partialSpectra)...),
    (NMRlab.coords(first(values(partialSpectra)),1), 
    collect(keys(partialSpectra))))  

@save "partial_spectra.jld2" partialSpects
partialSpects

### Normalisation

The partial spectra should be normalised in the sense that their integrals must be proportional to the number of spins contributing to each spectrum. Let us verify that this is indeed the case.

In [ ]:
partialSpectsInt = partialSpects |> Integral(1) 

TSPmax = maximum(partialSpectsInt[:,2] |> real)

n,m=size(partialSpectsInt)
a=plot()  # create an empty plot object to hold all spectra
for k in 1:m
    plot!(a,NMRlab.coords(partialSpectsInt[:,k],1)./600 .+ 4.78, real.(partialSpectsInt[:,k]) / TSPmax ,
    xaxis=:flip, ylims=10*[-0.1,1.0], label=NMRlab.coords(partialSpectsInt,2)[k],legend=:topright,size=(800,600),
    yticks=0:1.0:10.0,linewidth=2,
    xlabel="chemical shift (ppm)", ylabel="integrated intensity (spins)", title="simulated spectra of pure compounds")
end
a

### Alignment

The peak positions in the partial spectra must correspond to the ones in the experimental data, otherwise the determination of
amounts cannot be accurate. We can check this by examining the positions of certain prominent peaks that are easy to assign
in the experimental spectra. 

For the TSP peak, the alignment seems quite good, but not perfect in all cases of the experimental spectra:

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpects)[:,2], xaxis=:flip, 
    label=NMRlab.coords(partialSpects,2)[2],
    legend=:topleft)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, 
label="processed",xlims=0.0.+(-0.1,0.1),ylims=5e3 .* (-0.1,1.5))

We correct this by using the processor `PeakAlign`:

In [ ]:
processedData = processedData |> PeakAlign(1,-4.78*600,100) ;  # align the spectra to the TSP peak at 0 ppm

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpects)[:,2], xaxis=:flip, 
    label=NMRlab.coords(partialSpects,2)[2],
    legend=false)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, 
label="processed",xlims=0.0.+(-0.1,0.1),ylims=5e3 .* (-0.1,1.5))

Unfortunately, this is not the entire story. In the set of data we analyse here, there seems to be a systematic shift
of the peaks of most metabolites compared to the chemical shift values contained in the GISSMO database. This is visible for example in the
anomeric peak of $\alpha$-Glucose:

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpects)[:,3], xaxis=:flip, 
    label=NMRlab.coords(partialSpects,2)[2],
    legend=false)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, 
label="processed",xlims=0.0.+(5.1,5.3),ylims=5e3 .* (-0.1,1.5))

But also other metabolites, such as lactate, show the same misalignment, with a shift of the experimental
peaks about 0.03 ppm upfield:

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpects)[:,6], xaxis=:flip, 
    fmt=:svg,label=NMRlab.coords(partialSpects,2)[2],
    legend=false)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, fmt=:svg, 
label="processed",xlims=0.0.+(4.0,4.2),ylims=5e3 .* (-0.1,1.5))

In [ ]:
B = SpectData(partialSpects |> real |> pinv,(NMRlab.coords(partialSpects,2), NMRlab.coords(partialSpects,1)))

In [ ]:
rawamounts=SpectData(B  * processedData , (NMRlab.coords(B,1), NMRlab.coords(processedData,2)))

In [ ]:
Plots.bar(
     NMRlab.coords(rawamounts,1), 
     rawamounts[:,4], 
     xticks=(0.5:8, NMRlab.coords(rawamounts,1)),
     xlabel="metabolite", ylabel="relative amount (a.u.)", label=NMRlab.coords(rawamounts,2)[6], fmt=:svg)

In [ ]:
fittedSpectra = SpectData(partialSpects * rawamounts, (NMRlab.coords(partialSpects,1), NMRlab.coords(rawamounts,2)))
Plots.plot(NMRlab.coords(fittedSpectra,1)./600 .+ 4.78, real(fittedSpectra)[:,3], xaxis=:flip, fmt=:svg,label="fitted spectrum",legend=:topleft)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,17]), xaxis=:flip, fmt=:svg, 
label="measured spectrum",ylims=1e4 .* (-0.1,1.5),xlims=0.0.+(-0.1,5.7))

## Discussion

The above method to determine the amounts based on the partial spectra, being based on the Moore-Penrose pseudo-inverse, corresponds
to a least-squares fit of the experimental spectrum by a linear superposition of the partial spectra. This approach is simple and easy to
understand, but it suffers from a range of practical problems:

1. **Spectral overlap**: some of the peaks of the partial spectra overlap with each other
2. **Line shape**: the line width and shape of the partial spectra is artificially set by their processing
3. **Completeness**: the experimental spectrum contains many features that are not related to the partial spectra considered.
4. **Alignment**: the positions of some of the peaks may be slightly off, due to pH or solvent effects.

These shortcomings mean that the results depend on the list of metabolites that is used for analysis. Moreover, small
shifts in peak positions and spectral alignment lead to large changes in the resulting amounts. To use the above approach,
it is therefore very important to ensure good alignment, as well as narrowly defined conditions of measurement in terms
of pH, solvent, and temperature. This can be a problem, in particular when metabolites are involved that tend to 
shift the pH, e.g., lactic acid.

A possible alternative approach to least-squares minimization has been suggested by Domzal, Kazimierczuk et al. [*Anal. Chem.* 2024, **96**, 1, 188–196](https://doi.org/10.1021/acs.analchem.3c03594). Their
idea is to view the real part of the NMR spectrum as a *distribution* of intensity over the frequency domain. There *Wasserstein difference* $W_1(p,q)$ between
two distributions $p$ and $q$ corresponds to the minimial amount of "work" that needs to be done to transfer the contents of one distribution into the other. This metric
is different from the one we have used before in an important way. Let us assume that the spectra $p$ and $q$ both contain the same peak, but at different locations. The
least-squares difference we have used before is the same reqardless of how far the peak locations are from each other. The Wasserstein metric, by contrast, penalizes
larger deviations in position, and is more tolerant for small peak shifts. 

The Wasserstein metric is easy to understand if we view the two distributions as "piles" of dirt – it represents the minimum amount of "shovelling" that needs
to be carried out to convert one of them into the other. It is actually interesting that the mathematics of this problem go back to the [French mathematician Gaspard Monge](https://en.wikisource.org/wiki/1911_Encyclopædia_Britannica/Monge,_Gaspard), who
considered the construction and modification of military fortifications consisting of ditches and earthwalls. 

### Masking

One problem is that our list of partial spectra is not complete: there will always be signals in the experimental spectra from compounds that 
are not in the list. We define a mask that only lets pass parts of the spectrum that contain peaks that appear in the partial spectra.

In [ ]:
maskSpects = partialSpects .|> real .> 2.0 
mask=SpectData(reduce(|, maskSpects.dat, dims=2),(NMRlab.coords(maskSpects,1),["Mask"]))

In [ ]:
a=plot()  # create an empty plot object to hold all spectra
for (k,(name, spect)) in enumerate(partialSpectra)
    plot!(a,NMRlab.coords(spect,1)./600 .+ 4.78, real.(spect[:,1]).+(k-1)*4e2,
    xaxis=:flip, ylims=5.0e3.*[-0.1,1.0], xlims=(-0.1,8.0), fmt=:svg, label=name,legend=:topleft)
end
Plots.plot!(a, NMRlab.coords(mask,1)./600 .+ 4.78,  1e5*mask,fill=true, opacity=0.35, fmt=:svg)
a

In [ ]:
a=plot(NMRlab.coords(processedData,1)./600 .+ 4.78, processedData[:,3].*mask[:,1], 
     xaxis=:flip, fmt=:svg,label="masked measured spectrum",legend=:topleft, xlims=(-0.1,8.0), ylims=1e4 .* (-0.1,1.0) )
Plots.plot!(a, NMRlab.coords(mask,1)./600 .+ 4.78,  1e4*mask,fill=true, label="mask", opacity=0.15, fmt=:svg)

The masked spectrum can now be used to compute the integral. If we interpret the spectrum as a distribution, this is equivalent to the
cumulative distribution function (CDF).

In [ ]:
a=plot(NMRlab.coords(processedData,1)./600 .+ 4.78, processedData[:,3].*mask[:,1] |> Integral(1), 
     xaxis=:flip, fmt=:svg,label="masked measured spectrum",legend=:topleft, xlims=(-0.1,8.0), ylims=5e5 .* (-0.1,1.0) )

The same can now be done for the partial spectra:

In [ ]:
partialCDF = partialSpects |> real |> Integral(1) 
a=plot()  # create an empty plot object to hold all spectra
for k=1:size(partialCDF,2)
    plot!(a,NMRlab.coords(partialCDF[:,k],1)./600 .+ 4.78, real.(partialCDF[:,k]),
    xaxis=:flip, ylims=2.0e4.*[-0.1,1.0], xlims=(-0.1,8.0), fmt=:svg, label=NMRlab.coords(partialCDF,2)[k],legend=:topleft)
end
a

These partial CDF are then used to construct a linear superposition, just as we have done before for the
partial spectra. We obtain a CDF version of the $B$-matrix:

In [ ]:
Bcdf =  SpectData(pinv(partialCDF),(NMRlab.coords(partialCDF,2), NMRlab.coords(partialCDF,1)))

In [ ]:
processedCDF =  processedData .* mask |> real |> Integral(1) 
Plots.plot(NMRlab.coords(processedCDF,1)./600 .+ 4.78, processedCDF[:,:], xaxis=:flip, fmt=:svg,
    label="masked measured spectrum (CDF)",legend=false, xlims=(-0.1,8.0), ylims=5e5 .* (-0.1,1.0) )

In [ ]:
rawCDF = SpectData(circshift(Bcdf,(0,0))  * processedCDF , (NMRlab.coords(Bcdf,1), NMRlab.coords(processedCDF,2)))

In [ ]:
Plots.bar(
     NMRlab.coords(rawCDF,1), 
     rawCDF[:,1], 
     xticks=(0.5:8, NMRlab.coords(rawCDF,1)),
     xlabel="metabolite", ylabel="relative amount (a.u.)", label=NMRlab.coords(rawCDF,2)[6], fmt=:svg)

In [ ]:
Plots.plot(NMRlab.coords(processedCDF,1)./600 .+ 4.78, processedCDF[:,1], xaxis=:flip, fmt=:svg,
    label="masked measured spectrum (CDF)",legend=false, xlims=(-0.1,8.0), ylims=5e5 .* (-0.1,1.0) )

Plots.plot!(NMRlab.coords(processedCDF,1)./600 .+ 4.78, partialCDF * rawCDF[:,1], xaxis=:flip, fmt=:svg,
    label="reconstructed spectrum (CDF)",legend=:topleft, xlims=(-0.1,8.0), ylims=5e5 .* (-0.1,1.0) )

In [ ]:
function WassersteinDistance(spect1, spect2)
    # Placeholder function for computing Wasserstein distance between two spectra
    # In practice, this would involve computing the optimal transport plan between the two spectra
    return sum(abs.(spect1 - spect2))
end